# Lagrangian vs Eulerian Descriptions of Fluid Flow

## Overview

Classical fluid mechanics offers two fundamentally different ways to describe the motion of a fluid:

- The **Eulerian description** fixes attention on a point in space and asks "what is the velocity (or density, pressure, etc.) at this location at time $t$?" The unknown is a field $v(x, t)$ defined over space.
- The **Lagrangian description** follows individual fluid parcels and asks "where does a parcel that started at $x_0$ end up at time $t$?" The unknown is a map $X(x_0, t)$ satisfying $\dot{X} = v(X, t)$.

Both descriptions encode the same physics, but they illuminate different aspects. The connection between them is central to continuum mechanics, transport theory, and numerical methods.

## Eulerian Fields and Lagrangian Trajectories

Given a velocity field $v(x, t)$, the **Lagrangian trajectory** $X(t)$ of a particle starting at $X(0) = x_0$ satisfies the ODE:
$$
\frac{dX}{dt} = v(X(t), t), \quad X(0) = x_0
$$

## Material Derivative

The rate of change of a scalar quantity $f(x, t)$ along a fluid parcel trajectory is the **material (or convective) derivative**:
$$
\frac{Df}{Dt} = \frac{\partial f}{\partial t} + v \cdot \nabla f
$$
The term $v \cdot \nabla f$ is the **advection** term — it measures how much $f$ changes because the fluid carries the parcel to a new location.

## Conservation of Mass

The **continuity equation** (conservation of mass) in Eulerian form:
$$
\partial_t \rho + \nabla \cdot (\rho\, v) = 0
$$
In Lagrangian form, this becomes:
$$
\rho(X(t), t) \det(\nabla_x X(t)) = \rho_0(x_0)
$$
i.e., density is inversely proportional to the Jacobian determinant of the flow map.

## Divergence-Free Flow

For incompressible fluids, $\nabla \cdot v = 0$, which implies the flow map is **volume-preserving**: $\det(\nabla_x X) = 1$. A canonical incompressible 2D field is:
$$
v(x, y) = \left(-\sin(\pi x)\cos(\pi y),\; \cos(\pi x)\sin(\pi y)\right)
$$
One can verify $\partial_x v_1 + \partial_y v_2 = -\pi\cos(\pi x)\cos(\pi y) + \pi\cos(\pi x)\cos(\pi y) = 0$.

## What This Notebook Demonstrates

1. Eulerian view: streamlines and streamplot of the divergence-free field
2. Lagrangian view: particle trajectories integrated via RK4
3. Density transport: deformation of an initial circular patch of tracer
4. Comparison of linear advection vs nonlinear shear flows

### Setup

We implement a 4th-order Runge-Kutta (RK4) integrator for particle trajectories. The velocity field is given analytically, so we can evaluate it at any $(x, y, t)$ exactly.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# Divergence-free velocity field
def velocity(xy, t=0):
    """
    Evaluate v(x,y) = (-sin(pi*x)*cos(pi*y), cos(pi*x)*sin(pi*y)).
    xy: shape (N, 2) or (2,)
    Returns: same shape
    """
    xy = np.atleast_2d(xy)
    x, y = xy[:, 0], xy[:, 1]
    vx = -np.sin(np.pi * x) * np.cos(np.pi * y)
    vy = np.cos(np.pi * x) * np.sin(np.pi * y)
    return np.stack([vx, vy], axis=1)


def rk4_step(x, v_fn, t, dt):
    """One RK4 step for particle positions x (N, 2)."""
    k1 = v_fn(x, t)
    k2 = v_fn(x + 0.5 * dt * k1, t + 0.5 * dt)
    k3 = v_fn(x + 0.5 * dt * k2, t + 0.5 * dt)
    k4 = v_fn(x + dt * k3, t + dt)
    return x + (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def integrate_trajectories(X0, v_fn, t_end, dt):
    """
    Integrate N particles from X0 (N, 2) over [0, t_end] with step dt.
    Returns trajectory array of shape (n_steps+1, N, 2).
    """
    n_steps = int(t_end / dt)
    N = X0.shape[0]
    traj = np.zeros((n_steps + 1, N, 2))
    traj[0] = X0
    X = X0.copy()
    t = 0.0
    for k in range(n_steps):
        X = rk4_step(X, v_fn, t, dt)
        t += dt
        traj[k + 1] = X
    return traj


print("Velocity field and RK4 integrator defined.")

# Verify divergence-free
eps = 1e-5
xp = np.array([[0.3, 0.4]])
dvx_dx = (velocity(xp + np.array([[eps, 0]]))[:, 0] -
           velocity(xp - np.array([[eps, 0]]))[:, 0]) / (2 * eps)
dvy_dy = (velocity(xp + np.array([[0, eps]]))[:, 1] -
           velocity(xp - np.array([[0, eps]]))[:, 1]) / (2 * eps)
print(f"Divergence at (0.3, 0.4): {float((dvx_dx + dvy_dy)[0]):.2e} (should be ~0)")

### Eulerian View: Streamplot and Vorticity

The **streamlines** of a velocity field $v(x,t)$ at a fixed time $t$ are curves tangent to $v$ everywhere. They give an instantaneous picture of the flow geometry.

The **vorticity** $\omega = \partial_x v_y - \partial_y v_x$ measures local rotation. For our field:
$$
\omega(x,y) = \pi\sin(\pi x)\sin(\pi y) + \pi\sin(\pi x)\sin(\pi y) = 2\pi\sin(\pi x)\sin(\pi y)
$$
The vorticity alternates sign in a checkerboard pattern, forming a set of counter-rotating vortices.

In [2]:
# Grid for Eulerian visualization
xg = np.linspace(-1, 1, 40)
yg = np.linspace(-1, 1, 40)
XX, YY = np.meshgrid(xg, yg)
pts = np.stack([XX.ravel(), YY.ravel()], axis=1)
vv = velocity(pts)
VX = vv[:, 0].reshape(XX.shape)
VY = vv[:, 1].reshape(XX.shape)

# Vorticity: dvy/dx - dvx/dy
vort = 2 * np.pi * np.sin(np.pi * XX) * np.sin(np.pi * YY)
speed = np.sqrt(VX**2 + VY**2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Streamplot
ax = axes[0]
strm = ax.streamplot(xg, yg, VX, VY, color=speed,
                      cmap='plasma', linewidth=1.5, density=1.5)
fig.colorbar(strm.lines, ax=ax, label='Speed $|v|$')
ax.set_title('Eulerian: Streamlines of $v(x,y)$')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_aspect('equal')

# Vorticity field
ax = axes[1]
cf = ax.contourf(XX, YY, vort, levels=30, cmap='RdBu_r')
fig.colorbar(cf, ax=ax, label='Vorticity $\\omega$')
ax.quiver(XX[::4, ::4], YY[::4, ::4],
          VX[::4, ::4], VY[::4, ::4],
          alpha=0.6, scale=15, color='black', width=0.004)
ax.set_title('Vorticity $\\omega = \\partial_x v_y - \\partial_y v_x$')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_aspect('equal')

plt.suptitle('Eulerian Description of the Flow Field', fontsize=13)
plt.tight_layout()
plt.savefig("eulerian_view.png", dpi=80, bbox_inches='tight')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51923/2134380068.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Lagrangian View: Particle Trajectories via RK4

We now place a regular grid of particles and integrate their trajectories using RK4. Each particle follows $\dot{X} = v(X, t)$. For a steady (time-independent) field, particle paths coincide with streamlines; for time-dependent fields, they can differ dramatically.

We color each trajectory by the speed at its starting point, making it easy to identify fast and slow regions.

In [3]:
# Initial particle positions: regular grid in [-0.9, 0.9]^2
n_seed = 8
xs = np.linspace(-0.8, 0.8, n_seed)
ys = np.linspace(-0.8, 0.8, n_seed)
Xs_grid, Ys_grid = np.meshgrid(xs, ys)
X0 = np.stack([Xs_grid.ravel(), Ys_grid.ravel()], axis=1)

# Integrate trajectories
t_end = 3.0
dt = 0.02
traj = integrate_trajectories(X0, velocity, t_end, dt)
# traj shape: (n_steps+1, N, 2)

# Speed at initial positions
v0 = velocity(X0)
speed0 = np.linalg.norm(v0, axis=1)
norm_speed = (speed0 - speed0.min()) / (speed0.max() - speed0.min() + 1e-8)

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor('#111111')
fig.patch.set_facecolor('#111111')

# Background streamlines
ax.streamplot(xg, yg, VX, VY, color='white',
              linewidth=0.7, density=1.2)

cmap = plt.cm.plasma
for i in range(X0.shape[0]):
    path = traj[:, i, :]  # (n_steps+1, 2)
    # Plot trajectory with fading alpha
    color = cmap(norm_speed[i])
    ax.plot(path[:, 0], path[:, 1], '-', color=color,
            lw=0.8, alpha=0.7)
    ax.plot(path[0, 0], path[0, 1], 'o', color=color, ms=4, zorder=5)
    ax.plot(path[-1, 0], path[-1, 1], 's', color=color, ms=4, zorder=5)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_aspect('equal')
ax.set_title(f'Lagrangian: Particle Trajectories ($T={t_end}$, RK4)',
             color='white', fontsize=13)
ax.tick_params(colors='white')
for sp in ax.spines.values():
    sp.set_edgecolor('white')

plt.tight_layout()
plt.savefig("lagrangian_trajectories.png", dpi=80,
            bbox_inches='tight', facecolor='#111111')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51923/1276203611.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Density Transport: Deformation of a Tracer Patch

The **continuity equation** $\partial_t \rho + \nabla \cdot (\rho v) = 0$ governs how a density field $\rho(x,t)$ is transported by the flow. For an incompressible flow ($\nabla \cdot v = 0$), this simplifies to $D\rho/Dt = 0$: density is preserved along trajectories.

We initialize a circular patch of tracer (all particles inside a disk of radius $r_0$) and watch it deform. Because the flow is incompressible, the area of the patch is preserved — it can only be stretched and folded, never compressed or expanded.

In [4]:
# Dense tracer patch: circle of radius 0.3 centered at (0.3, 0.3)
rng = np.random.default_rng(0)
N_patch = 800
theta = rng.uniform(0, 2 * np.pi, N_patch)
r = 0.3 * np.sqrt(rng.uniform(0, 1, N_patch))
X_patch0 = np.stack([0.3 + r * np.cos(theta),
                      0.3 + r * np.sin(theta)], axis=1)

t_end_patch = 4.0
traj_patch = integrate_trajectories(X_patch0, velocity, t_end_patch, dt=0.02)

snapshot_times = [0, 0.5, 1.5, 3.0, 4.0]
snapshot_steps = [int(t / 0.02) for t in snapshot_times]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, step, t in zip(axes, snapshot_steps, snapshot_times):
    X_snap = traj_patch[step]
    ax.set_facecolor('#0d0d1a')
    # Background vorticity
    ax.contourf(XX, YY, vort, levels=20, cmap='RdBu', alpha=0.3)
    ax.scatter(X_snap[:, 0], X_snap[:, 1], s=4, alpha=0.8,
               color='gold', zorder=3)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.set_title(f'$t={t:.1f}$', fontsize=12)
    ax.tick_params(labelsize=8)

plt.suptitle('Density Transport: Deformation of Tracer Patch (Area Preserved)',
             fontsize=13)
plt.tight_layout()
plt.savefig("density_transport.png", dpi=80, bbox_inches='tight')
plt.show()

# Verify area conservation: convex hull area
from scipy.spatial import ConvexHull
hull0 = ConvexHull(X_patch0)
hull_final = ConvexHull(traj_patch[-1])
print(f"Convex hull area: t=0: {hull0.volume:.4f}, final: {hull_final.volume:.4f}")
print("(Area not exactly conserved for hull of scattered patch, but spread is bounded.)")

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51923/3963953910.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Convex hull area: t=0: 0.2697, final: 0.7969
(Area not exactly conserved for hull of scattered patch, but spread is bounded.)


### Linear Advection vs Nonlinear Shear

We now compare two flow types:

1. **Linear advection**: $v(x,y) = (c, 0)$ — uniform translation, no deformation
2. **Shear flow**: $v(x,y) = (y, 0)$ — shear in the $x$-direction proportional to $y$
3. **Our vortex field**: $v(x,y) = (-\sin\pi x \cos\pi y, \cos\pi x \sin\pi y)$ — nonlinear rotation

A circular tracer patch under linear advection remains circular; under shear it becomes an ellipse; under the vortex field it is folded into intricate filaments.

In [5]:
def v_advection(xy, t=0):
    return np.ones((len(xy), 2)) * np.array([0.3, 0.0])

def v_shear(xy, t=0):
    xy = np.atleast_2d(xy)
    vx = xy[:, 1].copy()
    vy = np.zeros(len(xy))
    return np.stack([vx, vy], axis=1)


# Small circle centered at origin
N_c = 500
theta = np.linspace(0, 2 * np.pi, N_c)
r_c = 0.25
X_circ = np.stack([r_c * np.cos(theta), r_c * np.sin(theta)], axis=1)

t_e = 2.0
traj_adv = integrate_trajectories(X_circ, v_advection, t_e, dt=0.02)
traj_shear = integrate_trajectories(X_circ, v_shear, t_e, dt=0.02)
traj_vortex = integrate_trajectories(X_circ, velocity, t_e, dt=0.02)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

trajs_list = [traj_adv, traj_shear, traj_vortex]
titles = [f'Linear advection $v=(0.3, 0)$\n$t={t_e}$',
          f'Shear $v=(y, 0)$\n$t={t_e}$',
          f'Vortex field\n$t={t_e}$']
colors_list = ['cornflowerblue', 'tomato', 'gold']

for ax, traj_cur, title, col in zip(axes, trajs_list, titles, colors_list):
    ax.set_facecolor('#111111')
    ax.plot(X_circ[:, 0], X_circ[:, 1], '--', color='gray', lw=1.5,
            alpha=0.5, label='Initial')
    X_final = traj_cur[-1]
    ax.plot(X_final[:, 0], X_final[:, 1], '-', color=col, lw=2,
            label='Final')
    # Show intermediate
    for frac in [0.25, 0.5, 0.75]:
        step = int(frac * len(traj_cur))
        ax.plot(traj_cur[step, :, 0], traj_cur[step, :, 1],
                '-', color=col, lw=0.8, alpha=0.4)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title(title, color='white', fontsize=11)
    ax.tick_params(colors='white')
    ax.legend(fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor('white')

plt.suptitle('Deformation of Circular Patch: Advection vs Shear vs Vortex',
             fontsize=13, color='white')
fig.patch.set_facecolor('#111111')
plt.tight_layout()
plt.savefig("deformation_comparison.png", dpi=80,
            bbox_inches='tight', facecolor='#111111')
plt.show()

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_51923/2138636795.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Interactive: Side-by-Side Eulerian/Lagrangian at Time $t$

The widget below lets you sweep the integration time $t$ with a slider. The left panel shows the Eulerian streamlines (which are constant for our steady field), while the right panel shows the current positions of Lagrangian particles (colored by their initial $x$-coordinate to make the deformation visible).

### Static Snapshot

This cell generates the snippet image: a three-panel figure showing the Eulerian streamlines, the Lagrangian tracer deformation, and the comparison of deformation types.

In [6]:
STATIC_SNAPSHOT = True

if STATIC_SNAPSHOT:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    fig_s, axes_s = plt.subplots(1, 3, figsize=(13, 4))
    fig_s.patch.set_facecolor('#111111')

    # Panel 1: Eulerian streamlines
    ax = axes_s[0]
    ax.set_facecolor('#111111')
    strm = ax.streamplot(xg, yg, VX, VY, color=speed,
                         cmap='plasma', linewidth=1.5, density=1.5)
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.set_title('Eulerian Streamlines', color='white', fontsize=11)
    ax.tick_params(colors='white')

    # Panel 2: Lagrangian tracer at t=3
    ax = axes_s[1]
    ax.set_facecolor('#111111')
    step_snap = int(3.0 / 0.02)
    X_snap2 = traj_patch[min(step_snap, len(traj_patch)-1)]
    ax.contourf(XX, YY, vort, levels=20, cmap='RdBu', alpha=0.25)
    ax.scatter(X_snap2[:, 0], X_snap2[:, 1], s=5, alpha=0.9, color='gold')
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.set_title('Lagrangian Tracer ($t=3$)', color='white', fontsize=11)
    ax.tick_params(colors='white')

    # Panel 3: Shear vs vortex deformation
    ax = axes_s[2]
    ax.set_facecolor('#111111')
    ax.plot(X_circ[:, 0], X_circ[:, 1], '--', color='gray', lw=1.5, alpha=0.5)
    ax.plot(traj_shear[-1, :, 0], traj_shear[-1, :, 1], '-',
            color='tomato', lw=2, label='Shear')
    ax.plot(traj_vortex[-1, :, 0], traj_vortex[-1, :, 1], '-',
            color='gold', lw=2, label='Vortex')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.set_title('Patch Deformation', color='white', fontsize=11)
    ax.legend(fontsize=9)
    ax.tick_params(colors='white')

    for ax in axes_s:
        for sp in ax.spines.values():
            sp.set_edgecolor('white')

    plt.suptitle('Lagrangian vs Eulerian Flow Description',
                 color='white', fontsize=13)
    plt.tight_layout()
    plt.savefig("snippet.png", dpi=100, bbox_inches='tight',
                facecolor=fig_s.get_facecolor())
    plt.close()
    print("snippet.png saved.")

snippet.png saved.


## Takeaways

- The **Eulerian description** represents the flow as a field $v(x,t)$ at fixed spatial coordinates; streamlines and vorticity are natural Eulerian objects.
- The **Lagrangian description** tracks particle trajectories $X(t)$ satisfying $\dot{X} = v(X,t)$; it makes individual parcel histories explicit.
- The **material derivative** $Df/Dt = \partial_t f + v \cdot \nabla f$ connects the two frames: it measures change as seen by a moving parcel.
- For **incompressible flows** ($\nabla \cdot v = 0$), the Jacobian of the flow map is 1 — area/volume of material regions is exactly conserved, even as they are stretched and folded.
- **RK4 integration** of the particle ODE is far more accurate than Euler's method for moderate time steps, capturing the spiraling structure of vortical flows faithfully.
- Different flow types produce qualitatively different deformation patterns: linear advection translates without distortion, shear stretches, and vortex fields fold and filament material patches.

## Bibliography

- **Batchelor, G. K.** (2000). *An Introduction to Fluid Dynamics.* Cambridge University Press.
- **Chorin, A. J. & Marsden, J. E.** (1990). *A Mathematical Introduction to Fluid Mechanics.* Springer.
- **Ottino, J. M.** (1989). *The Kinematics of Mixing: Stretching, Chaos, and Transport.* Cambridge University Press.
- **LeVeque, R. J.** (2002). *Finite Volume Methods for Hyperbolic Problems.* Cambridge University Press.
- **Shadden, S. C., Lekien, F. & Marsden, J. E.** (2005). *Definition and properties of Lagrangian coherent structures from finite-time Lyapunov exponents.* Physica D, 212(3-4), 271–304.